# Climate-Health Modeling Runner

This notebook runs the full modeling + sensitivity pipeline and writes a run summary to `{OUT_ROOT}/modeling/`.


In [ ]:
# ── BOUNDARY VERSION SELECTOR ──────────────────────────────────────────────
# Choose which HSA boundary bundle to use for this analysis.
# Must match a bundle produced by HSA_FINAL.ipynb.
#   'v6'  — original greedy algorithm (no post-selection corrections)
#   'v7'  — + anchor upgrade/demotion + major-orphan promotion
#   'v8'  — + satellite bubble boundaries
BOUNDARY_VERSION = "v7"   # change as needed
# ────────────────────────────────────────────────────────────────────────────


In [ ]:
# ============================================================================
# CONFIGURATION - Full modeling + sensitivity pipeline
# ============================================================================
from pathlib import Path
import os
import subprocess
import sys
import time
import json

# --- Core Settings ---
NETWORK = 'INF'                        # Network type: INF, NCD
HSA_MODE = 'footprint'                 # HSA optimization mode
TARGET_COL = 'diarrheal_count_adjusted' if NETWORK == 'INF' else 'hypertension_count_adjusted'

# --- Reproducibility ---
RANDOM_SEED = 42

# --- Paths ---
PIPELINE_VERSION = os.environ.get('PIPELINE_VERSION', 'v7')
OUT_ROOT = Path(os.environ.get('HSA_OUT_DIR', os.environ.get('PIPELINE_OUT_DIR', f'out_{PIPELINE_VERSION}')))
MODELING_DIR = OUT_ROOT / 'modeling'
SENSITIVITY_DIR = OUT_ROOT / 'sensitivity'
TEXT_RESULTS_DIR = OUT_ROOT / 'textresults'

INPUT_CSV = MODELING_DIR / f'{NETWORK}_{HSA_MODE}_modeling_dataset.csv'
OUTPUT_PREFIX = f'{NETWORK}_{HSA_MODE}'

COMPREHENSIVE_DIR = MODELING_DIR / 'results_comprehensive'
PARSIMONIOUS_DIR = MODELING_DIR / 'results_parsimonious'
ML_RESULTS_DIR = MODELING_DIR / 'results_ml'
IMPROVED_RESULTS_DIR = MODELING_DIR / 'results_improved'
ANOMALIES_DIR = MODELING_DIR / 'results_anomalies'

for d in [
    MODELING_DIR,
    SENSITIVITY_DIR,
    TEXT_RESULTS_DIR,
    COMPREHENSIVE_DIR,
    PARSIMONIOUS_DIR,
    ML_RESULTS_DIR,
    IMPROVED_RESULTS_DIR,
    ANOMALIES_DIR,
]:
    d.mkdir(parents=True, exist_ok=True)

SCRIPT_RUN_LOG = []


def run_step(step_name, cmd):
    print(f"\n{'=' * 80}")
    print(f"STEP: {step_name}")
    print('Running:', ' '.join(cmd))
    print(f"{'=' * 80}")

    start_time = time.time()
    result = subprocess.run(cmd, check=False)
    duration_sec = round(time.time() - start_time, 2)

    step_result = {
        'step': step_name,
        'command': cmd,
        'returncode': int(result.returncode),
        'duration_seconds': duration_sec,
    }
    SCRIPT_RUN_LOG.append(step_result)

    if result.returncode != 0:
        raise RuntimeError(f"{step_name} failed with code {result.returncode}")

    print(f"Completed {step_name} in {duration_sec}s")


print('=' * 80)
print('CLIMATE-HEALTH PIPELINE - CONFIGURATION')
print('=' * 80)
print(f'Network: {NETWORK}')
print(f'HSA mode: {HSA_MODE}')
print(f'Target column: {TARGET_COL}')
print(f'Random seed: {RANDOM_SEED}')
print(f'Input CSV: {INPUT_CSV}')
print(f'Output prefix: {OUTPUT_PREFIX}')
print(f'Modeling output root: {MODELING_DIR}')
print(f'Sensitivity output root: {SENSITIVITY_DIR}')
print('=' * 80)


In [ ]:
# ============================================================================
# EXECUTION - Run all modeling and sensitivity scripts
# ============================================================================
RUN_PREP = False  # Set True to rerun climate_health_modeling.py data prep + EDA

if RUN_PREP:
    run_step('climate_health_modeling.py', [
        sys.executable, 'climate_health_modeling.py',
        '--network', NETWORK,
        '--hsa-mode', HSA_MODE,
        '--input-csv', str(INPUT_CSV),
        '--target-col', TARGET_COL,
        '--output-dir', str(MODELING_DIR),
        '--output-prefix', OUTPUT_PREFIX,
    ])

modeling_steps = [
    (
        'climate_health_modeling_comprehensive.py',
        [
            sys.executable, 'climate_health_modeling_comprehensive.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--target-col', TARGET_COL,
            '--input-csv', str(INPUT_CSV),
            '--output-dir', str(COMPREHENSIVE_DIR),
            '--output-prefix', OUTPUT_PREFIX,
            '--random-seed', str(RANDOM_SEED),
        ],
    ),
    (
        'climate_health_modeling_parsimonious.py',
        [
            sys.executable, 'climate_health_modeling_parsimonious.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--target-col', TARGET_COL,
            '--input-csv', str(INPUT_CSV),
            '--output-dir', str(PARSIMONIOUS_DIR),
            '--output-prefix', OUTPUT_PREFIX,
            '--random-seed', str(RANDOM_SEED),
        ],
    ),
    (
        'train_ml_models.py',
        [
            sys.executable, 'train_ml_models.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--target-col', TARGET_COL,
            '--data-dir', str(MODELING_DIR),
            '--output-dir', str(ML_RESULTS_DIR),
            '--random-seed', str(RANDOM_SEED),
        ],
    ),
    (
        'train_improved_models.py',
        [
            sys.executable, 'train_improved_models.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--target-col', TARGET_COL,
            '--data-dir', str(MODELING_DIR),
            '--output-dir', str(IMPROVED_RESULTS_DIR),
            '--random-seed', str(RANDOM_SEED),
        ],
    ),
    (
        'climate_health_modeling_anomalies.py',
        [
            sys.executable, 'climate_health_modeling_anomalies.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--target-col', TARGET_COL,
            '--input-csv', str(INPUT_CSV),
            '--output-dir', str(ANOMALIES_DIR),
            '--output-prefix', OUTPUT_PREFIX,
            '--random-seed', str(RANDOM_SEED),
        ],
    ),
]

for step_name, cmd in modeling_steps:
    run_step(step_name, cmd)

sensitivity_steps = [
    (
        '08_climate_ar_decomposition.py',
        [
            sys.executable, '08_climate_ar_decomposition.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--input-csv', str(INPUT_CSV),
            '--target-col', TARGET_COL,
            '--output-dir', str(SENSITIVITY_DIR / 'analysis_variance_decomposition'),
            '--text-output-dir', str(TEXT_RESULTS_DIR),
        ],
    ),
    (
        '09_spatial_unit_comparison.py',
        [
            sys.executable, '09_spatial_unit_comparison.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--data-dir', 'data',
            '--out-dir', str(OUT_ROOT),
            '--output-dir', str(SENSITIVITY_DIR / 'analysis_spatial_comparison'),
            '--text-output-dir', str(TEXT_RESULTS_DIR),
            '--target-col', TARGET_COL,
        ],
    ),
    (
        '10_spatial_autocorrelation.py',
        [
            sys.executable, '10_spatial_autocorrelation.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--data-dir', 'data',
            '--out-dir', str(OUT_ROOT),
            '--output-dir', str(SENSITIVITY_DIR / 'analysis_spatial_autocorrelation'),
            '--target-col', TARGET_COL,
        ],
    ),
    (
        '11_gravity_sensitivity_analysis.py',
        [
            sys.executable, '11_gravity_sensitivity_analysis.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--data-dir', 'data',
            '--out-dir', str(OUT_ROOT),
            '--output-dir', str(SENSITIVITY_DIR / 'analysis_gravity_sensitivity'),
            '--text-output-dir', str(TEXT_RESULTS_DIR),
            '--target-col', TARGET_COL,
        ],
    ),
    (
        '12_extreme_event_analysis.py',
        [
            sys.executable, '12_extreme_event_analysis.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--out-dir', str(OUT_ROOT),
            '--output-dir', str(SENSITIVITY_DIR / 'analysis_extreme_events'),
            '--text-output-dir', str(TEXT_RESULTS_DIR),
            '--target-col', TARGET_COL,
        ],
    ),
    (
        '13_exclusion_analysis.py',
        [
            sys.executable, '13_exclusion_analysis.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--data-dir', 'data',
            '--out-dir', str(OUT_ROOT),
            '--output-dir', str(SENSITIVITY_DIR / 'analysis_exclusion'),
            '--text-output-dir', str(TEXT_RESULTS_DIR),
        ],
    ),
    (
        '14_climate_exclusion_test.py',
        [
            sys.executable, '14_climate_exclusion_test.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--data-dir', 'data',
            '--out-dir', str(OUT_ROOT),
            '--output-dir', str(SENSITIVITY_DIR / 'analysis_climate_exclusion'),
            '--text-output-dir', str(TEXT_RESULTS_DIR),
        ],
    ),
    (
        '15_weight_sensitivity_analysis.py',
        [
            sys.executable, '15_weight_sensitivity_analysis.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--out-dir', str(OUT_ROOT),
            '--output-dir', str(SENSITIVITY_DIR / 'analysis_weight_sensitivity'),
            '--text-output-dir', str(TEXT_RESULTS_DIR),
        ],
    ),
    (
        '16_within_hsa_heterogeneity.py',
        [
            sys.executable, '16_within_hsa_heterogeneity.py',
            '--network', NETWORK,
            '--hsa-mode', HSA_MODE,
            '--data-dir', 'data',
            '--out-dir', str(OUT_ROOT),
            '--output-dir', str(SENSITIVITY_DIR / 'analysis_climate_heterogeneity'),
            '--text-output-dir', str(TEXT_RESULTS_DIR),
        ],
    ),
]

for step_name, cmd in sensitivity_steps:
    run_step(step_name, cmd)

run_log_path = MODELING_DIR / f'{OUTPUT_PREFIX}_pipeline_run_log.json'
with open(run_log_path, 'w') as f:
    json.dump(SCRIPT_RUN_LOG, f, indent=2)

print(f"\nSaved run log to: {run_log_path}")


In [ ]:
# ============================================================================
# SUMMARY - Write consolidated markdown summary of generated outputs
# ============================================================================
from datetime import datetime

summary_path = TEXT_RESULTS_DIR / f'run_climate_health_modeling_{NETWORK}_{HSA_MODE}_results.md'
run_log_path = MODELING_DIR / f'{OUTPUT_PREFIX}_pipeline_run_log.json'

if run_log_path.exists():
    run_log = json.loads(run_log_path.read_text())
else:
    run_log = SCRIPT_RUN_LOG


def collect_files(base_dir, max_files=12):
    base = Path(base_dir)
    if not base.exists():
        return []
    files = [p for p in base.rglob('*') if p.is_file()]
    files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return files[:max_files]


modeling_dirs = [
    COMPREHENSIVE_DIR,
    PARSIMONIOUS_DIR,
    ML_RESULTS_DIR,
    IMPROVED_RESULTS_DIR,
    ANOMALIES_DIR,
]

sensitivity_dirs = [
    SENSITIVITY_DIR / 'analysis_variance_decomposition' / NETWORK,
    SENSITIVITY_DIR / 'analysis_spatial_comparison' / NETWORK,
    SENSITIVITY_DIR / 'analysis_spatial_autocorrelation' / NETWORK,
    SENSITIVITY_DIR / 'analysis_gravity_sensitivity' / NETWORK,
    SENSITIVITY_DIR / 'analysis_extreme_events' / NETWORK,
    SENSITIVITY_DIR / 'analysis_exclusion',
    SENSITIVITY_DIR / 'analysis_climate_exclusion',
    SENSITIVITY_DIR / 'analysis_weight_sensitivity' / NETWORK,
    SENSITIVITY_DIR / 'analysis_climate_heterogeneity' / NETWORK,
]

lines = []
lines.append('# Climate-Health Pipeline Summary')
lines.append('')
lines.append(f'- Generated: {datetime.now().isoformat(timespec="seconds")}')
lines.append(f'- Network: {NETWORK}')
lines.append(f'- HSA mode: {HSA_MODE}')
lines.append(f'- Target column: {TARGET_COL}')
lines.append(f'- Input dataset: `{INPUT_CSV}`')
lines.append('')

lines.append('## Step Execution Status')
lines.append('')
if run_log:
    for item in run_log:
        status = 'OK' if item.get('returncode', 1) == 0 else f"FAILED ({item.get('returncode')})"
        lines.append(f"- `{item['step']}`: {status}, {item.get('duration_seconds', 'n/a')}s")
else:
    lines.append('- No run log found in-memory or on disk.')
lines.append('')

lines.append('## Modeling Outputs (`{OUT_ROOT}/modeling`)')
lines.append('')
for d in modeling_dirs:
    files = collect_files(d)
    all_files = [p for p in d.rglob('*') if p.is_file()] if d.exists() else []
    lines.append(f"### `{d}`")
    lines.append(f"- File count: {len(all_files)}")
    if files:
        for p in files:
            lines.append(f"- `{p}`")
    else:
        lines.append('- No files found')
    lines.append('')

lines.append('## Sensitivity Outputs (`{OUT_ROOT}/sensitivity`)')
lines.append('')
for d in sensitivity_dirs:
    files = collect_files(d)
    all_files = [p for p in d.rglob('*') if p.is_file()] if d.exists() else []
    lines.append(f"### `{d}`")
    lines.append(f"- File count: {len(all_files)}")
    if files:
        for p in files:
            lines.append(f"- `{p}`")
    else:
        lines.append('- No files found')
    lines.append('')

summary_path.write_text(''.join(lines))
print(f'Saved summary to: {summary_path}')


In [ ]:
# AUTO_NOTEBOOK_SUMMARY_V1
from pathlib import Path
from datetime import datetime
import os
import json

NOTEBOOK_NAME = "run_climate_health_modeling"
NETWORK = globals().get('NETWORK', os.environ.get('NETWORK', 'INF'))
HSA_MODE = globals().get('HSA_MODE', os.environ.get('HSA_MODE', ''))

suffix = f"{NETWORK}_{HSA_MODE}" if HSA_MODE else f"{NETWORK}"

out_root = Path(globals().get('OUT_DIR', globals().get('OUT_ROOT', os.environ.get('HSA_OUT_DIR', os.environ.get('PIPELINE_OUT_DIR', f"out_{os.environ.get('PIPELINE_VERSION', 'v7')}")))))
summary_dir = out_root / 'textresults'
summary_dir.mkdir(parents=True, exist_ok=True)
summary_path = summary_dir / f"{NOTEBOOK_NAME}_{suffix}_results.md"

files = [p for p in out_root.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv', '.json', '.md', '.txt', '.png', '.pdf', '.geojson', '.parquet'}]
files.sort(key=lambda p: p.stat().st_mtime, reverse=True)
important = files[:120]

NL = "\n"

blocks = []
blocks.append(f"# Notebook Results: {NOTEBOOK_NAME}")

meta = [
    f"- Generated: {datetime.now().isoformat(timespec='seconds')}",
    f"- Network: {NETWORK}",
]
if HSA_MODE:
    meta.append(f"- HSA mode: {HSA_MODE}")
blocks.append(NL.join(meta))

file_lines = ['## Important Output Files', '']
for p in important:
    file_lines.append(f"- `{p}`")
blocks.append(NL.join(file_lines))

nb_path = Path(f"{NOTEBOOK_NAME}.ipynb")
if nb_path.exists():
    try:
        nb_data = json.loads(nb_path.read_text())
        blocks.append('## Displayed Cell Outputs')

        for idx, cell in enumerate(nb_data.get('cells', []), start=1):
            if cell.get('cell_type') != 'code':
                continue
            outputs = cell.get('outputs', []) or []
            if not outputs:
                continue

            blocks.append(f"### Cell {idx}")

            for out in outputs:
                otype = out.get('output_type')

                if otype == 'stream':
                    text = ''.join(out.get('text', [])) if isinstance(out.get('text', []), list) else str(out.get('text', ''))
                    text = text.rstrip()
                    if text:
                        blocks.append("```text" + NL + text + NL + "```")

                elif otype in ('execute_result', 'display_data'):
                    data = out.get('data', {})
                    if 'text/markdown' in data:
                        md = ''.join(data['text/markdown']) if isinstance(data['text/markdown'], list) else str(data['text/markdown'])
                        md = md.rstrip()
                        if md:
                            blocks.append(md)
                    elif 'text/plain' in data:
                        txt = ''.join(data['text/plain']) if isinstance(data['text/plain'], list) else str(data['text/plain'])
                        txt = txt.rstrip()
                        if txt:
                            blocks.append("```text" + NL + txt + NL + "```")
                    elif 'text/html' in data:
                        html = ''.join(data['text/html']) if isinstance(data['text/html'], list) else str(data['text/html'])
                        html = html.rstrip()
                        if html:
                            blocks.append("```html" + NL + html + NL + "```")
                    elif 'image/png' in data or 'image/jpeg' in data:
                        blocks.append('_[Image output omitted in text summary]_')

                elif otype == 'error':
                    tb = out.get('traceback', []) or []
                    if tb:
                        err = NL.join(str(t) for t in tb)
                    else:
                        err = f"{out.get('ename', 'Error')}: {out.get('evalue', '')}"
                    blocks.append("```text" + NL + err + NL + "```")

    except Exception as e:
        blocks.append('## Displayed Cell Outputs')
        blocks.append(f"Could not parse notebook outputs: {e}")

summary = (NL + NL).join(b for b in blocks if b and str(b).strip()) + NL
summary_path.write_text(summary)
print(f"Saved notebook summary: {summary_path}")
